# TUGAS MANDIRI PRAKTIKUM BIG DATA — PERTEMUAN 4
## Pengenalan PySpark DataFrame & Operasi Terdistribusi

| Informasi | Detail |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Pertemuan** | 4 (Tugas Mandiri) |
| **Nama Mahasiswa** | Yarsya |
| **NPM** | 2505060066 |
| **Program Studi** | S1 Teknologi Informasi — Universitas Tidar |

---

### Skenario Tugas
Seluruh proses analisis data transaksi e-commerce bermigrasi dari library pandas ke **Apache Spark (PySpark)** untuk mengantisipasi pertumbuhan volume data yang melampaui kapasitas RAM komputer tunggal. Analisis dilakukan dengan membaca data langsung dari HDFS (`hdfs://localhost:9000/...`), menangani nilai *missing*, melakukan transformasi kolom, agregasi bisnis, serta menyimpan kembali hasilnya ke HDFS.

## 0. Inisialisasi SparkSession
Membuka koneksi utama `SparkSession` dalam mode *local[*]* menggunakan seluruh core CPU yang tersedia di sistem.

In [1]:
import os
import sys

# Memastikan worker Spark menggunakan binary python environment yang sama
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, avg, sum, round

spark = SparkSession.builder \
    .appName("Tugas4-PySpark-Yarsya") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("SparkSession berhasil diinisialisasi!")
print("Versi Spark:", spark.version)

26/11/05 17:12:33 WARN Utils: Your hostname, meil resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/11/05 17:12:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/05 17:12:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession berhasil diinisialisasi!
Versi Spark: 3.5.9


## 1. Menyiapkan Dataset Transaksi & Unggah ke HDFS
Dataset periode September 2026 (1.000 transaksi) digenerasi dan diunggah ke path HDFS `/user/mahasiswa/tugas4/transaksi_september_2026.csv`.

In [2]:
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4_local = pd.DataFrame(data)
df_tugas4_local.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset lokal berhasil dibuat: {df_tugas4_local.shape[0]} baris")

# Unggah ke HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/mahasiswa/tugas4/
!hdfs dfs -ls -h /user/mahasiswa/tugas4/transaksi_september_2026.csv

Dataset lokal berhasil dibuat: 1000 baris


-rw-r--r--   1 meil supergroup     81.5 K 2026-11-05 17:12 /user/mahasiswa/tugas4/transaksi_september_2026.csv


---
## Bagian A. Membaca dan Eksplorasi Awal (Bobot 15%)

Membaca dataset **langsung dari HDFS** menggunakan URI `hdfs://localhost:9000/...` dengan `header=True` dan `inferSchema=True`, lalu menampilkan skema kolom, jumlah baris (`count()`), dan 10 baris pertama.

In [3]:
# Membaca data langsung dari HDFS ke Spark DataFrame
hdfs_path = "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv"
df_spark = spark.read.csv(hdfs_path, header=True, inferSchema=True)

# 1. Tampilkan skema DataFrame
print("=== SKEMA DATAFRAME ===")
df_spark.printSchema()

# 2. Tampilkan jumlah baris
total_baris = df_spark.count()
print(f"Total jumlah baris di HDFS: {total_baris}")

# 3. Tampilkan 10 baris pertama
print("\n=== 10 BARIS PERTAMA ===")
df_spark.show(10)

=== SKEMA DATAFRAME ===
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)



Total jumlah baris di HDFS: 1000

=== 10 BARIS PERTAMA ===
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|
|ORD-3004|2026-09-10 00:00:00|        Rumah Tangga|Yogyakarta|          10|       60000|         E-Wallet|   4.0|
|ORD-3005|2026-09-09 00:00:00

---
## Bagian B. Menangani Data Kosong (Bobot 15%)

Mengecek keberadaan nilai kosong (`null`/`NaN`) pada kolom `rating`, lalu menanganinya menggunakan fungsi `df.na.fill()` beserta argumen penjelasan rasional.

In [4]:
# Menghitung jumlah nilai kosong pada kolom rating
jumlah_null_rating = df_spark.filter(col("rating").isNull()).count()
print(f"Jumlah nilai null/kosong pada kolom rating: {jumlah_null_rating} baris ({jumlah_null_rating / total_baris * 100:.1f}%)")

# Penanganan data kosong menggunakan df.na.fill() dengan nilai 0 (unreviewed)
df_clean = df_spark.na.fill({"rating": 0})

# Verifikasi bahwa kolom rating sudah tidak memiliki nilai null
sisa_null = df_clean.filter(col("rating").isNull()).count()
print(f"Sisa nilai null setelah na.fill: {sisa_null}")
df_clean.show(5)

Jumlah nilai null/kosong pada kolom rating: 204 baris (20.4%)
Sisa nilai null setelah na.fill: 0
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|
|ORD-3004|2026-09-10 00:00:00|        Rumah Tangga|Yogyakarta|          10|       60000|         E-Wallet

### Penjelasan Pemilihan `df.na.fill({"rating": 0})`
Metode `df.na.fill({"rating": 0})` dipilih dibandingkan `df.na.drop()` karena:
1. Nilai kosong pada kolom rating merepresentasikan transaksi yang diselesaikan oleh pelanggan namun **belum diberikan ulasan (unreviewed)**, bukan data transaksi yang rusak atau tidak valid.
2. Jika menggunakan `drop()`, kita akan kehilangan sekitar 20% baris transaksi (202 data), yang akan mendistorsi perhitungan finansial (`total_pendapatan` dan `unit_terjual`).
3. Dengan mengisi rating bernilai `0`, seluruh catatan transaksi tetap utuh untuk analisis omzet bisnis, sekaligus memudahkan pemfilteran ulasan aktual (`rating > 0`) saat mengukur kepuasan pelanggan.

---
## Bagian C. Transformasi Data (Bobot 20%)

Menambahkan dua kolom turunan baru pada DataFrame:
1. `total_pendapatan` = `unit_terjual * harga_satuan`
2. `tier_transaksi` = bernilai `"Besar"` jika `total_pendapatan > 500000`, atau `"Kecil"` jika sebaliknya.

In [5]:
# Menambahkan kolom total_pendapatan dan tier_transaksi
df_transform = df_clean \
    .withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan")) \
    .withColumn("tier_transaksi", when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil"))

print("10 Baris Hasil Transformasi Data:")
df_transform.select("order_id", "kota", "kategori", "unit_terjual", "harga_satuan", "total_pendapatan", "tier_transaksi").show(10)

10 Baris Hasil Transformasi Data:


+--------+----------+--------------------+------------+------------+----------------+--------------+
|order_id|      kota|            kategori|unit_terjual|harga_satuan|total_pendapatan|tier_transaksi|
+--------+----------+--------------------+------------+------------+----------------+--------------+
|ORD-3000|Yogyakarta|        Rumah Tangga|           3|       90000|          270000|         Kecil|
|ORD-3001|      Solo|   Makanan & Minuman|           3|      200000|          600000|         Besar|
|ORD-3002|  Semarang|Kesehatan & Kecan...|           8|       60000|          480000|         Kecil|
|ORD-3003|  Semarang|   Makanan & Minuman|           6|      350000|         2100000|         Besar|
|ORD-3004|Yogyakarta|        Rumah Tangga|          10|       60000|          600000|         Besar|
|ORD-3005| Purworejo|             Fashion|           5|       20000|          100000|         Kecil|
|ORD-3006|Yogyakarta|   Makanan & Minuman|           2|       20000|           40000|      

---
## Bagian D. Analisis dengan GroupBy (Bobot 30%)

Menjawab tiga pertanyaan bisnis menggunakan fungsi agregasi PySpark DataFrame API (`groupBy`, `agg`, `orderBy`).

In [6]:
# 1. Kategori apa yang memiliki total_pendapatan tertinggi?
print("=== 1. TOTAL PENDAPATAN PER KATEGORI (URUT DARI TERTINGGI) ===")
df_kat_pendapatan = df_transform.groupBy("kategori") \
    .agg(sum("total_pendapatan").alias("total_pendapatan")) \
    .orderBy(col("total_pendapatan").desc())

df_kat_pendapatan.show()
top_kat = df_kat_pendapatan.first()
print(f"-> Kategori dengan total pendapatan tertinggi adalah '{top_kat['kategori']}' dengan Rp {top_kat['total_pendapatan']:,}")

=== 1. TOTAL PENDAPATAN PER KATEGORI (URUT DARI TERTINGGI) ===


+--------------------+----------------+
|            kategori|total_pendapatan|
+--------------------+----------------+
|        Rumah Tangga|       138665000|
|   Makanan & Minuman|       131890000|
|Kesehatan & Kecan...|       128595000|
|            Olahraga|       126650000|
|             Fashion|       124075000|
|          Elektronik|       110295000|
+--------------------+----------------+



-> Kategori dengan total pendapatan tertinggi adalah 'Rumah Tangga' dengan Rp 138,665,000


In [7]:
# 2. Kota mana dengan jumlah transaksi tier 'Besar' terbanyak?
print("=== 2. JUMLAH TRANSAKSI TIER 'BESAR' PER KOTA ===")
df_kota_besar = df_transform.filter(col("tier_transaksi") == "Besar") \
    .groupBy("kota") \
    .agg(count("order_id").alias("jumlah_transaksi_besar")) \
    .orderBy(col("jumlah_transaksi_besar").desc())

df_kota_besar.show()
top_kota = df_kota_besar.first()
print(f"-> Kota dengan jumlah transaksi tier 'Besar' terbanyak adalah '{top_kota['kota']}' ({top_kota['jumlah_transaksi_besar']} transaksi)")

=== 2. JUMLAH TRANSAKSI TIER 'BESAR' PER KOTA ===


+----------+----------------------+
|      kota|jumlah_transaksi_besar|
+----------+----------------------+
|      Solo|                    92|
|  Magelang|                    78|
|   Kebumen|                    78|
|Yogyakarta|                    75|
| Purworejo|                    66|
|  Semarang|                    65|
+----------+----------------------+

-> Kota dengan jumlah transaksi tier 'Besar' terbanyak adalah 'Solo' (92 transaksi)


In [8]:
# 3. Berapa rata-rata rating untuk masing-masing metode_pembayaran?
# Catatan: Dihitung baik untuk seluruh data (termasuk rating 0) dan khusus ulasan aktif (rating > 0)
print("=== 3. RATA-RATA RATING PER METODE PEMBAYARAN (ULASAN AKTIF > 0) ===")
df_avg_rating = df_transform.filter(col("rating") > 0) \
    .groupBy("metode_pembayaran") \
    .agg(
        round(avg("rating"), 2).alias("rata_rata_rating"),
        count("order_id").alias("jumlah_ulasan")
    ) \
    .orderBy(col("rata_rata_rating").desc())

df_avg_rating.show()

=== 3. RATA-RATA RATING PER METODE PEMBAYARAN (ULASAN AKTIF > 0) ===


+-----------------+----------------+-------------+
|metode_pembayaran|rata_rata_rating|jumlah_ulasan|
+-----------------+----------------+-------------+
|              COD|            4.17|          203|
|    Transfer Bank|            4.16|          203|
|         E-Wallet|            4.14|          199|
|     Kartu Kredit|            4.11|          191|
+-----------------+----------------+-------------+



---
## Bagian E. Menyimpan Hasil ke HDFS (Bobot 20%)

Menyimpan DataFrame hasil transformasi (Bagian C) ke HDFS dalam format CSV baru pada path `/user/mahasiswa/tugas4/hasil_olahan_csv` dan memverifikasi isi direktori.

In [9]:
# Menyimpan DataFrame hasil olahan ke HDFS dalam format CSV
output_hdfs_path = "hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_olahan_csv"
df_transform.write.mode("overwrite").option("header", "true").csv(output_hdfs_path)

print("DataFrame berhasil ditulis ke HDFS:", output_hdfs_path)

DataFrame berhasil ditulis ke HDFS: hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_olahan_csv


In [10]:
# Verifikasi berkas hasil penulisan di HDFS
!hdfs dfs -ls -h /user/mahasiswa/tugas4/hasil_olahan_csv

Found 2 items
-rw-r--r--   3 meil supergroup          0 2026-11-05 17:12 /user/mahasiswa/tugas4/hasil_olahan_csv/_SUCCESS
-rw-r--r--   3 meil supergroup     95.0 K 2026-11-05 17:12 /user/mahasiswa/tugas4/hasil_olahan_csv/part-00000-0593942a-f3fa-47ae-bdda-3a6c90476515-c000.csv


### Penjelasan Berkas Partisi (`part-00000...`)
Ketika Apache Spark menyimpan DataFrame ke media penyimpanan (seperti HDFS atau disk lokal), direktori keluaran berisi beberapa berkas pecahan bertulisan `part-00000-...`, `part-00001-...`, dan berkas status `_SUCCESS`.

Hal ini terjadi karena:
1. **Komputasi dan I/O Terdistribusi Secara Paralel:** Data dipecah menjadi beberapa *partitions* di dalam memori cluster/core CPU. Masing-masing task pekerja (*executor/worker thread*) memproses dan menuliskan pecahan datanya sendiri-sendiri secara simultan ke disk tanpa harus berkumpul di satu simpul driver (*bottleneck elimination*).
2. **Skalabilitas Data Skala Terabyte:** Untuk data berukuran ratusan gigabyte hingga terabyte, menuliskan satu berkas raksasa secara serial akan sangat lambat dan berisiko gagal. Sistem penyimpanan terdistribusi memanfaatkan berkas-berkas partisi paralel agar proses baca-tulis dapat diskalakan secara horizontal (*linear horizontal scalability*).
3. **Flag `_SUCCESS`:** Berkas kosong penanda transaksi bahwa seluruh partisi dari job penulisan tersebut telah tuntas secara atomik tanpa kegagalan di tengah jalan.